# DepthWizard v4 — GAMUS (PNG mirror), Kaggle 2×T4

**Accelerator `GPU T4 x2` · Internet ON · Add-ons → Secrets: `HF_TOKEN` *and*
`GITHUB_PAT`.**

`DepthWizard` is a private repo, so the clone needs a GitHub personal access
token — a fine-grained token with *Contents: read* on this one repo, or a
classic token with `repo`. The notebook clones into `/kaggle/temp`, not
`/kaggle/working`: `git clone` stores the token-bearing URL in `.git/config`,
and `/kaggle/working` is published as the notebook's output dataset. It also
resets `origin` to the clean URL, and scrubs the token out of any clone error
before printing it.
Start the real run with **Save Version → Save & Run All (Commit)** — the headless
commit gets the full ~12 h and persists `/kaggle/working`; an interactive session
idles out long before the run ends.

Attach the GAMUS PNG dataset (`{train,val,test}/{images,depth}/gamus_*.png`) as
the only input. Nothing is downloaded: `prepare_data.py` pulls one HDF5 per tile
from the gated `earthflow/GAMUS` repo, but the attached mirror is already
unpacked, so cell 1 packs those PNGs straight into the memmap shard store that
`train.py` reads.

**Set `SMOKE = True` for the first run** (~5 min). Success is a clean exit, *one*
set of loss lines in the log — which is what proves the rank-1 stdout redirect
worked — and `best.pt` / `metrics.json` written once.

### The one thing to check in the output
The `HEIGHT CHECK` line. A PNG mirror's height encoding cannot be assumed, so
the packer guesses from the dtype and prints what it produced. GAMUS AGL should
read median ~2–5 m, p99 ~30–60 m, max ≲100 m. If everything is 100× too large it
is centimetres — set `DEPTH_SCALE = "0.01"` and re-run.

### Two things that differ from the H100 profile
* **No class masks** ship with the PNG mirror, so every pixel packs as
  `NO_LABEL`. `seg_ce_loss` short-circuits an all-unlabelled batch to zero, so
  the 0.2-weight semantic head contributes nothing; per-class metrics in
  `metrics.json` come out empty. Depth training is unaffected.
* **SynRS3D is dropped** (`--datasets gamus`), so every source tile is 0.33 m
  GSD. The 0.30–1.20 m jitter still applies, capped per tile by
  `achievable_gsd_range`.


In [ ]:
# ═══ DepthWizard v4 — GAMUS PNG mirror on Kaggle 2×T4 — run this one cell ═══
import os, sys, subprocess, pathlib

REPO   = "/kaggle/temp/DepthWizard"   # NOT /kaggle/working: a clone of a private repo
                                      # carries the PAT in .git/config, and /kaggle/working
                                      # is published as the notebook's output dataset
CODE   = f"{REPO}/FineTunning/V4_Kaggle"
GAMUS  = "/kaggle/input"          # autodetects the directory holding train/images
DATA   = "/kaggle/temp/dwdata"    # packed store — off the 20 GB output quota
OUT    = "/kaggle/working/outputs/v4"
SMOKE  = False                    # True first: ~5 min, proves DDP + data + model end-to-end
RESUME = ""                       # session 2: "/kaggle/input/<prev>/outputs/v4/last_full.pt"
DEPTH_SCALE = "auto"              # "auto" | "0.01" (cm→m) | "1.0" (already metres)
TRAIN_TILES = 0                   # 0 = all; lower it if /kaggle/temp fills
MAX_MINUTES = 480                 # total training budget across ALL sessions

GIT_URL = "github.com/AbhayKale332/DepthWizard.git"
sh = lambda c: subprocess.run(c, shell=True, check=True)

def clone():
    """The repo is private, so this needs GITHUB_PAT (Add-ons -> Secrets, `repo` scope).

    List form and captured output, never an f-string through the shell: a failing
    clone prints the URL it tried, and that URL contains the token.
    """
    pat = ""
    try:
        from kaggle_secrets import UserSecretsClient
        pat = UserSecretsClient().get_secret("GITHUB_PAT")
    except Exception as e:
        print("no GITHUB_PAT secret (", e, ") - trying an anonymous clone")
    url = f"https://x-access-token:{pat}@{GIT_URL}" if pat else f"https://{GIT_URL}"
    r = subprocess.run(["git", "clone", "--depth", "1", url, REPO],
                       capture_output=True, text=True)
    if r.returncode:
        raise SystemExit("git clone failed: "
                         + (r.stderr.replace(pat, "***") if pat else r.stderr))
    # `git clone` writes the token-bearing URL into .git/config. Drop it.
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin",
                    f"https://{GIT_URL}"], check=True)
    print("cloned", REPO)

if not os.path.isdir(REPO):
    clone()
os.chdir(CODE); sys.path.insert(0, CODE)
sh("pip install -q -r requirements.txt")

try:                               # the DINOv3-SAT encoder is gated too
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print("!! no HF_TOKEN secret:", e)

# ── 1. PNG tree → memmap shard store ───────────────────────────────────────
# train.py only ever reads the packed store; prepare_data.py builds it from the
# gated HDF5 repo, which is not what is attached here. Same store, local PNGs.
PACKER = pathlib.Path(CODE, "pack_gamus_png.py")
if not PACKER.exists():
    PACKER.write_text(r'''
import sys, numpy as np, cv2
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent))
from dwdata.packed import ShardWriter, PackedStore, store_exists

SRC, OUT, SCALE, LIMIT = sys.argv[1], Path(sys.argv[2]), sys.argv[3], int(sys.argv[4])
EXT = (".png", ".tif", ".tiff", ".jpg", ".jpeg")

def find_root(base):
    base = Path(base)
    if (base / "train" / "images").is_dir():
        return base
    for p in sorted(base.rglob("train/images")):
        return p.parent.parent
    raise SystemExit(f"no <root>/train/images under {base}")

def rd(p):
    a = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)   # UNCHANGED: 16-bit must not truncate
    if a is None:
        raise OSError(f"unreadable {p}")
    return a

def rgb_of(p):
    a = rd(p)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, 2)
    a = a[..., :3][..., ::-1]                      # cv2 gives BGR
    if a.dtype != np.uint8:
        a = a.astype(np.float32) / max(1.0, float(a.max())) * 255.0
    return np.ascontiguousarray(a.astype(np.uint8))

def sq(a, t):
    h, w = a.shape[:2]
    if h < t or w < t:
        return None
    return a[(h - t) // 2:(h - t) // 2 + t, (w - t) // 2:(w - t) // 2 + t]

def pack(split_dir, out, limit):
    if store_exists(out):
        print(f"[{out.name}] already packed"); return
    imgs = {p.stem: p for p in sorted((split_dir / "images").iterdir()) if p.suffix.lower() in EXT}
    deps = {p.stem: p for p in sorted((split_dir / "depth").iterdir()) if p.suffix.lower() in EXT}
    pairs = [(s, imgs[s], deps[s]) for s in imgs if s in deps]
    if not pairs:
        raise SystemExit(f"no image/depth pairs under {split_dir}")
    pairs.sort()
    if limit and limit < len(pairs):
        rng = np.random.default_rng(0)
        pairs = [pairs[i] for i in sorted(rng.permutation(len(pairs))[:limit].tolist())]
    h, w = rd(pairs[0][1]).shape[:2]
    T = min(h, w) - min(h, w) % 16                 # the model's 16 px patch grid
    d0 = rd(pairs[0][2])
    scale = (float(SCALE) if SCALE != "auto" else
             (1.0 if d0.dtype != np.uint16 else
              (0.01 if float(np.percentile(d0.astype(np.float32), 99.9)) > 1000 else 1.0)))
    print(f"[{out.name}] {len(pairs)} pairs  tile_px={T}  depth dtype={d0.dtype} x{scale:g} -> m", flush=True)
    wtr, done, skip, samp = ShardWriter(out, tile_px=T, gsd_m=0.33, shard_tiles=128, has_seg=False), 0, 0, []
    for stem, ip, dp in pairs:
        try:
            r, d = rgb_of(ip), rd(dp)
            if d.ndim == 3:
                d = d[..., 0]
            if d.shape[:2] != r.shape[:2]:
                # onto the image grid BEFORE cropping, or the two crops are different ground.
                # Nearest: heights must not be interpolated across a roof edge.
                d = cv2.resize(d, (r.shape[1], r.shape[0]), interpolation=cv2.INTER_NEAREST)
            r, d = sq(r, T), sq(d, T)
            if r is None or d is None:
                skip += 1; continue
            agl = d.astype(np.float32) * scale
            val = np.isfinite(agl) & (agl > -2.0) & (agl < 500.0)
            wtr.add(stem, r, np.where(val, np.clip(agl, 0, None), 0.0), None, val)
            if len(samp) < 200 and val.any():
                samp.append(agl[val][::97])
            done += 1
        except Exception as e:
            print(f"  skip {stem}: {e}"); skip += 1
        if done and done % 500 == 0:
            print(f"  {done}/{len(pairs)}", flush=True)
    idx = wtr.finalise()
    print(f"[{out.name}] packed {idx['n']} tiles, {skip} skipped")
    if samp:
        s = np.concatenate(samp)
        print(f"[{out.name}] HEIGHT CHECK  min {s.min():.2f}  median {np.median(s):.2f} "
              f" p99 {np.percentile(s,99):.2f}  max {s.max():.2f} m   <-- GAMUS AGL is ~0-100 m")

root = find_root(SRC)
print("source:", root)
for split, lim in (("train", LIMIT), ("val", 0)):
    if (root / split).is_dir():
        pack(root / split, OUT / "gamus" / split, lim)
# The 2/98 stretch bounds are a full-tile histogram; no training worker should pay one.
for idx in sorted((OUT / "gamus").glob("*/index.json")):
    st = PackedStore(idx.parent)
    st.prime_stretch_bounds(2.0, 98.0, workers=4)
    print("  stretch bounds primed:", idx.parent, len(st), "tiles")
''')
sh(f"python pack_gamus_png.py {GAMUS} {DATA} {DEPTH_SCALE} {TRAIN_TILES}")
sh(f"du -sh {DATA}; df -h /kaggle/temp | tail -1")

# ── 2. Train: torchrun, one process per T4 ──────────────────────────────
os.environ.update(PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
                  NCCL_P2P_DISABLE="1",          # no NVLink on Kaggle's T4 pair
                  OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
args = (f"--data_root {DATA} --output_dir {OUT} --datasets gamus "
        # Turing has no bf16 tensor cores -> fp16 + GradScaler; ViT-L needs
        # encoder checkpointing to fit 15 GB (static VRAM is ~8.3 GB before activations).
        "--amp_dtype fp16 --grad_checkpoint_encoder true "
        "--batch_size 2 --grad_accum 6 --eval_batch_mult 8 "      # 2 ranks x 2 x 6 = batch 24
        "--num_workers 2 --prefetch_factor 2 --compile_model false "
        f"--epochs 40 --max_minutes {MAX_MINUTES} --session_minutes 480 "
        "--save_full_state true --make_zip false")                # the zip blows the 20 GB quota
if SMOKE:
    args = f"--smoke --data_root {DATA} --output_dir {OUT} --datasets gamus --make_zip false"
if RESUME:
    args += f" --resume {RESUME}"
sh(f"torchrun --standalone --nnodes=1 --nproc_per_node=2 train.py {args}")


## If the session dies

`last_full.pt` carries the optimiser, scaler, EMA, teacher, RNG and elapsed
minutes. Attach the dead run's version as an input to a fresh copy of this
notebook and set `RESUME = "/kaggle/input/<name>/outputs/v4/last_full.pt"`.
`run.log` should show the LR continuing *down* the cosine, not re-warming.
Keep `MAX_MINUTES` the same in both sessions — it is the total budget that
drives the cosine; `session_minutes` is what caps one session.

`/kaggle/temp` is wiped between sessions, so a resumed run re-clones the repo
and re-packs the PNGs first (a few minutes). That is deliberate: the store does not belong in
`/kaggle/working`, where it would eat the 20 GB output quota.

If training finished but the exports did not, run `bash run_kaggle.sh finalize`
in a short single-GPU notebook against the committed `best.pt`.
